In [ ]:
# Imports
import pandas as pd
import numpy as np
import requests
import time

# Load in data
df = pd.read_csv('./data/TransactionReport.csv')

# Define constants
LATITUDE = 38.318056
LONGITUDE = -77.509722
TIMEZONE = 'America/New_York'

# Dictionary mapping WMO weather codes to readable conditions
WEATHER_CODES = {
    0: 'Clear sky', 1: 'Mainly clear', 2: 'Partly cloudy', 3: 'Overcast',
    45: 'Foggy', 48: 'Foggy',
    51: 'Light drizzle', 53: 'Moderate drizzle', 55: 'Dense drizzle',
    61: 'Slight rain', 63: 'Moderate rain', 65: 'Heavy rain',
    71: 'Slight snow', 73: 'Moderate snow', 75: 'Heavy snow', 77: 'Snow grains',
    80: 'Rain showers', 81: 'Rain showers', 82: 'Rain showers',
    85: 'Snow showers', 86: 'Snow showers',
    95: 'Thunderstorm', 96: 'Thunderstorm', 99: 'Thunderstorm'
}

# Dictionary mapping WMO weather codes to simplified categories
WEATHER_CATEGORY = {
    0: 'Clear', 1: 'Clear', 2: 'Cloudy', 3: 'Cloudy',
    45: 'Cloudy', 48: 'Cloudy',
    51: 'Rainy', 53: 'Rainy', 55: 'Rainy',
    61: 'Rainy', 63: 'Rainy', 65: 'Rainy',
    71: 'Snowy', 73: 'Snowy', 75: 'Snowy', 77: 'Snowy',
    80: 'Rainy', 81: 'Rainy', 82: 'Rainy',
    85: 'Snowy', 86: 'Snowy',
    95: 'Severe', 96: 'Severe', 99: 'Severe'
}

# Function to fetch weather data for a given data with retry logic
def get_weather_data(lat, lon, date, max_retries = 5):
    """
    Fetch weather data with automatic retry on timeout
    Uses exponential backoff: 1s, 2s, 4s, 8s, 16s
    """

    # API endpoint
    url = "https://archive-api.open-meteo.com/v1/archive"

    # Parameters for API request
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': date,
        'end_date': date,
        'hourly': 'temperature_2m,relative_humidity_2m,precipitation,weather_code',
        'timezone': TIMEZONE
    }
    
    # Attempt to fetch data with retries
    for attempt in range(max_retries):
        try:
            # Poll API
            response = requests.get(url, params = params, timeout = 15)
            response.raise_for_status()
            return response.json()
        except Exception as e:
            # If failure, print error and retry after delay (runs up to 5 times)
            if attempt < max_retries - 1:
                print(f"  (retry {attempt + 1}/{max_retries - 1})", end = ' ')
                time.sleep(1)
            else:
                print(f"✗")
                return None

# Function to append weather data to DataFrame
def append_weather_to_dataframe(df):
    """
    Append gametime weather conditions to DataFrame
    
    Adds columns:
    - weather_code: WMO weather code
    - weather_condition: Human-readable condition
    - temperature_c: Temperature in Celsius at gametime
    - temperature_f: Temperature in Fahrenheit at gametime
    - precipitation: Precipitation amount at gametime
    - humidity: Humidity % at gametime
    - weather_category: Simplified category (Clear, Cloudy, Rainy, Snowy, Severe)
    """

    # Convert game times to DateTime objects
    df['Event Timestamp'] = pd.to_datetime(df['Event Timestamp'])
    
    # Initialize new columns
    df['weather_code'] = None
    df['weather_condition'] = None
    df['temperature_c'] = None
    df['temperature_f'] = None
    df['precipitation'] = None
    df['humidity'] = None
    df['weather_category'] = None
    
    # Get unique dates from the DataFrame to ensure only calling API once per date
    unique_dates = df['Event Timestamp'].dt.strftime('%Y-%m-%d').unique()
    weather_cache = {}
    
    # Fetch weather data for each unique date and store in cache
    print(f"Fetching weather for {len(unique_dates)} unique dates...")
    for i, date in enumerate(unique_dates):
        print(f"  [{i+1}/{len(unique_dates)}] {date}", end=' ')
        weather_cache[date] = get_weather_data(LATITUDE, LONGITUDE, date)
        print("✓")
        time.sleep(0.3)
    
    # Append weather data to each row based on gametime
    print("\nProcessing rows")

    # Iterate through each row in the DataFrame
    for idx, row in df.iterrows():
        # Extract date and hour from gametime
        date_str = row['Event Timestamp'].strftime('%Y-%m-%d')
        game_hour = row['Event Timestamp'].hour
        
        # Get weather data for this row's date from the cache
        weather_data = weather_cache.get(date_str)

        # If weather data exists, append it to the DataFrame
        if weather_data and 'hourly' in weather_data:
            # Extract hourly data
            hourly = weather_data['hourly']
            
            # Check if the game hour is within the range of available hourly data
            if game_hour < len(hourly['temperature_2m']):
                # Get weather code and temperature for the game hour
                code = hourly['weather_code'][game_hour]
                temp_c = hourly['temperature_2m'][game_hour]
                
                # Append weather data to the DataFrame
                df.at[idx, 'weather_code'] = code
                df.at[idx, 'weather_condition'] = WEATHER_CODES.get(code, 'Unknown')
                df.at[idx, 'temperature_c'] = temp_c
                df.at[idx, 'temperature_f'] = np.round((temp_c * 9/5) + 32, 1) if temp_c is not None else None
                df.at[idx, 'precipitation'] = hourly['precipitation'][game_hour]
                df.at[idx, 'humidity'] = hourly['relative_humidity_2m'][game_hour]
                df.at[idx, 'weather_category'] = WEATHER_CATEGORY.get(code, 'Unknown')
        
        # Print progress every 500
        if (idx + 1) % 500 == 0:
            print(f"  Processed {idx + 1} rows...")
    
    # Print on completion. Return augmented DataFrame
    print(f"\nDone! Added weather data to {len(df)} rows.")
    return df

# Append weather data to DataFrame and save to new CSV
df = append_weather_to_dataframe(df)
df.to_csv('./data/TransactionReportWithWeather.csv', index = False)

Fetching weather for 66 unique dates...
  [1/66] 2025-04-08 ✓
  [2/66] 2025-04-09 ✓
  [3/66] 2025-04-10 ✓
  [4/66] 2025-04-11 ✓
  [5/66] 2025-04-12 ✓
  [6/66] 2025-04-13 ✓
  [7/66] 2025-04-22 ✓
  [8/66] 2025-04-23 ✓
  [9/66] 2025-04-24 ✓
  [10/66] 2025-04-25 ✓
  [11/66] 2025-04-26 ✓
  [12/66] 2025-04-27 ✓
  [13/66] 2025-05-06 ✓
  [14/66] 2025-05-07 ✓
  [15/66] 2025-05-08 ✓
  [16/66] 2025-05-09 ✓
  [17/66] 2025-05-10 ✓
  [18/66] 2025-05-11 ✓
  [19/66] 2025-05-20 ✓
  [20/66] 2025-05-21 ✓
  [21/66] 2025-05-22 ✓
  [22/66] 2025-05-23 ✓
  [23/66] 2025-05-24 ✓
  [24/66] 2025-05-25 ✓
  [25/66] 2025-06-03 ✓
  [26/66] 2025-06-04 ✓
  [27/66] 2025-06-05 ✓
  [28/66] 2025-06-06 ✓
  [29/66] 2025-06-07   (retry 1/4)   (retry 2/4)   (retry 3/4) ✓
  [30/66] 2025-06-08 ✓
  [31/66] 2025-06-24 ✓
  [32/66] 2025-06-25   (retry 1/4) ✓
  [33/66] 2025-06-26 ✓
  [34/66] 2025-06-27 ✓
  [35/66] 2025-06-28 ✓
  [36/66] 2025-06-29 ✓
  [37/66] 2025-07-04 ✓
  [38/66] 2025-07-05 ✓
  [39/66] 2025-07-06 ✓
  [40/66] 2025-0